```{index} Types
```

# Types


```{contents}
:local:
:depth: 2
```


A type is a contract for values and operations, not just a memory-size label. A quantity, a measured duration, a price, and a customer identifier have different meanings even if each looks numeric. Choose a type that preserves the information and calculations your requirement needs.


```{index} int; long; double; decimal; bool; char; string
```

## Choose a Type for the Data



| Data | Typical type here | Reason |
| --- | --- | --- |
| Cases ordered | `int` | Whole-number count |
| A very large accumulated count | `long` | Wider whole-number range |
| Measured duration or distance | `double` | Approximate fractional measurement |
| Unit price and invoice amount | `decimal` | Decimal arithmetic appropriate to this invoice |
| Shipping qualification | `bool` | `true` or `false` |
| A one-letter category | `char` | One UTF-16 code unit, often one basic character |
| Customer name or postal code | `string` | Text, including significant leading zeros |

A postal code such as `"00123"` is an identifier, not a quantity to add or multiply. Storing it as an integer would discard the leading zeros. Type choice should follow meaning, not appearance.


```{index} integer range; byte; overflow; checked
```

### Integer Range and Overflow



`int` uses 32 bits and ranges from -2,147,483,648 through 2,147,483,647. `long` uses 64 bits and has a wider range. `byte` ranges from zero through 255. Unsigned types have no negative values; they do not validate every business rule about a count. For example, an `int` can represent a negative order quantity even when the business prohibits it.

Digit separators make large literals readable; an uppercase `L` suffix explicitly makes a literal a `long`. Each statement below prints information that can be compared with the declared range.


In [1]:
int quantity = 12;
long annualUnits = 3_000_000_000L;
byte categoryCode = 200;
Console.WriteLine(quantity);
Console.WriteLine(annualUnits);
Console.WriteLine(categoryCode);
Console.WriteLine(int.MaxValue);


12
3000000000
200
2147483647


**Overflow** occurs when a calculation's result does not fit its numeric type. `checked(...)` requests an exception for overflowing integral arithmetic; `unchecked(...)` permits the specified integral wraparound behavior. Neither behavior makes an incorrect business total acceptable.


In [1]:
int maximum = int.MaxValue;
Console.WriteLine(unchecked(maximum + 1));
long wider = (long)maximum + 1;
Console.WriteLine(wider);


-2147483648
2147483648


The outputs are -2,147,483,648 and 2,147,483,648. The cast in the second calculation occurs **before** addition, so the addition uses `long`. An explicit `checked(maximum + 1)` would throw an overflow exception; it is not executed in this example. Chapter 6 develops failure handling. Prefer checked arithmetic or a deliberately wider type when a silent wrapped value would be harmful.


```{index} float; double; decimal; numeric literal suffix
```

## Fractional Numbers and Money



A real-number literal without a suffix, such as `38.50`, is a `double`. Use `m` for `decimal`, `f` for `float`, and optionally `d` for `double`. `float` and `double` use binary floating-point representations; many decimal fractions are approximations. `decimal` exactly represents many ordinary decimal fractions, including 0.1, within its finite range and precision.[^1]

This book uses `decimal` for invoice amounts. It still needs a rounding policy: one third has no finite decimal expansion, and calculations can overflow. “Use decimal” is not a promise that every financial calculation is exact.


In [1]:
using System.Globalization;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;

Console.WriteLine(0.1 + 0.2 == 0.3);
Console.WriteLine(0.1m + 0.2m == 0.3m);
decimal unitPrice = 38.50m;
int quantity = 12;
decimal subtotal = quantity * unitPrice;
Console.WriteLine(subtotal);


False
True
462.00


The comparisons print `False` and `True`; the subtotal is 462.00 in this fixed-format culture. The small binary discrepancy matters when testing equality. Later numerical problems require tolerance choices appropriate to their scale. Do not mix `decimal` and `double` arithmetic without a deliberate conversion.

Some examples set `CultureInfo.CurrentCulture` to `InvariantCulture` so printed decimal separators are reproducible. Culture is a set of formatting conventions. Input and Output develops its role; the setup is repeated so each cell runs independently.


```{index} Math.Round; midpoint rounding; currency rounding
```

### Round a Business Result Deliberately



Specify **when** to round and **how** to treat a value halfway between two choices. In this example, round the tax to cents and then add it to the subtotal. The sample rate is a classroom assumption, not a statement about tax law.

`Math.Round(value, 2, MidpointRounding.AwayFromZero)` rounds to two fractional places and sends an exact midpoint away from zero. The default midpoint rule is `ToEven`, which picks the even final digit. Neither rule is universally correct for every business requirement.


In [1]:
using System.Globalization;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;

decimal amount = 10.10m;
decimal sampleTaxRate = 0.05m;
decimal rawTax = amount * sampleTaxRate;
decimal roundedTax = Math.Round(rawTax, 2, MidpointRounding.AwayFromZero);
Console.WriteLine(rawTax);
Console.WriteLine(roundedTax);
Console.WriteLine(Math.Round(rawTax, 2));
Console.WriteLine(amount + roundedTax);


0.5050
0.51
0.50
10.61


The raw tax is 0.5050, the chosen rule produces 0.51, the default rule produces 0.50, and the total using the chosen rule is 10.61. A two-place display format alone would not change the stored tax; rounding assigns an explicitly rounded value. Avoid accidentally adding unrounded tax while showing rounded tax on the invoice.


In [ ]:
// ### Exercise: Choose and Round Invoice Types
// Use int quantity 3, decimal unitPrice 10.10m, and taxRate 0.05m. Calculate subtotal, tax rounded to cents away from zero, and total. Print each. Expected total: 31.82.
// Your code starts here.

// Your code ends here.


In [1]:
using System.Globalization;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;

// ### Solution
int quantity = 3;
decimal unitPrice = 10.10m;
decimal taxRate = 0.05m;
decimal subtotal = quantity * unitPrice;
decimal tax = Math.Round(subtotal * taxRate, 2, MidpointRounding.AwayFromZero);
decimal total = subtotal + tax;
Console.WriteLine(subtotal);
Console.WriteLine(tax);
Console.WriteLine(total);


30.30
1.52
31.82


```{index} bool; UTF-16; char; string
```

(string)=

## Boolean and Text Values



A `bool` holds `true` or `false`; printing it produces `True` or `False`. It has two possible logical values, which does not mean it universally occupies one physical bit. Comparisons produce Boolean values, as you will see in Operators.

A `char` is one 16-bit **UTF-16 code unit**. A `string` contains a sequence of those units. Many basic characters use one unit, while some Unicode symbols use two and some displayed characters combine multiple symbols. Therefore, string `.Length` is not always a count of the characters a reader sees.[^2]


In [1]:
bool paid = false;
char category = 'A';
string postalCode = "00123";
string symbol = "😀";
Console.WriteLine(paid);
Console.WriteLine(category);
Console.WriteLine(postalCode);
Console.WriteLine(symbol.Length);


False
A
00123
2


```{index} string immutability; string assignment
```

### String Values Are Immutable



**Immutable** means an existing string's contents cannot be changed. Assigning another string to a variable replaces the reference stored in that variable; it does not edit the old string. Calling `Replace` or `ToUpper` returns a result that you must use or assign.


In [1]:
string customer = "Riverside Dental";
string original = customer;
customer = customer.Replace("Dental", "Clinic");
Console.WriteLine(original);
Console.WriteLine(customer);


Riverside Dental
Riverside Clinic


```{index} string indexing; Length; string methods
```

(more-string-methods)=

### Indexing, Length, and Small String Operations



An **index** identifies a position starting at zero. For `"PAPER"`, indexes zero through four identify `P`, `A`, `P`, `E`, and `R`. The last valid index is `Length - 1`. Indexing an empty string at zero or any string at `Length` would fail at runtime.

`.Length` is a property, a value you read without parentheses. `.Contains(...)` and `.Replace(...)` are methods, operations called with parentheses and arguments. You do not need class design to use these library members.


In [1]:
string item = "PAPER";
Console.WriteLine(item.Length);
Console.WriteLine(item[0]);
Console.WriteLine(item[item.Length - 1]);
Console.WriteLine(item.Contains("APE"));
Console.WriteLine(item.Replace("PAPER", "TONER"));


5
P
R
True
TONER


```{index} string escapes; verbatim string; concatenation
```

(string-concatenation)=

### Literal Escapes and Concatenation



Double quotes delimit string literals; single quotes delimit `char` literals. In an ordinary string, `\n` creates a newline, `\t` creates a tab, `\\` represents a backslash, and `\"` represents a double quote. A **verbatim string** starts with `@` and treats backslashes literally.

**Concatenation** joins text with `+`. When a string is involved, numbers may be converted to text rather than added. Read the order carefully.


In [1]:
Console.WriteLine("Quantity: " + 12);
Console.WriteLine("Total: " + 2 + 3);
Console.WriteLine("Total: " + (2 + 3));
Console.WriteLine("Invoice\nEnd of report");
Console.WriteLine(@"C:\Orders\invoice.txt");


Quantity: 12
Total: 23
Total: 5
Invoice
End of report
C:\Orders\invoice.txt


The second line displays `Total: 23`; the third displays `Total: 5`. Parentheses complete the numeric addition before the string concatenation. The path example prints a path as text; it does not create or read a file.


```{index} value type; reference type; default
```

## Assignment Semantics and Default Values



Numeric types, `bool`, and `char` are **value types**. Assigning a simple value such as an integer copies that value. `string` is a **reference type**: assignment copies a reference to an object. String immutability does not change that classification. Other reference types can refer to mutable shared objects, studied in later chapters.

Avoid the shortcut “value types are always on the stack.” A value can be stored as part of another object; physical storage location is not the definition of assignment semantics.


In [1]:
int quantity = 12;
int copiedQuantity = quantity;
copiedQuantity = 20;
Console.WriteLine(quantity);
Console.WriteLine(copiedQuantity);
string customer = "Riverside";
string copiedCustomer = customer;
Console.WriteLine(object.ReferenceEquals(customer, copiedCustomer));
copiedCustomer = "Lakeside";
Console.WriteLine(customer);
Console.WriteLine(copiedCustomer);


12
20
True
Riverside
Lakeside


`ReferenceEquals` checks whether the two references identify the same object. The check prints `True` before reassignment. The strings are never modified in place.

`default(int)` is zero, `default(bool)` is false, and a reference type's default is `null`, meaning no object. This does not permit reading an unassigned local variable. `void`, seen in Chapter 1's `Main`, indicates that a method returns no value; it is not a variable type.


```{index} implicit conversion; explicit conversion; cast
```

## Numeric Conversions Require Judgment



An **implicit conversion** is permitted without cast syntax. An **explicit conversion** requires a written conversion such as `(int)measurement`. Permission and precision are separate questions: `int` to `long` preserves every integer, but an allowed `int` to `float` conversion can lose precision. A cast is not input validation.


In [1]:
int exactCount = 16_777_217;
long wider = exactCount;
float approximate = exactCount;
Console.WriteLine(wider);
Console.WriteLine((int)approximate);
double measured = -3.9;
int whole = (int)measured;
Console.WriteLine(whole);


16777217
16777216
-3


The `long` preserves 16,777,217; the `float` round trip gives 16,777,216. Casting -3.9 to `int` gives -3, truncating toward zero rather than rounding to the nearest integer. `Math.Round` returns a rounded numeric result without turning a `double` into an `int`.

A cast such as `(int)"12"` cannot interpret digit text as a number. **Parsing** examines text according to numeric syntax; the Input and Output section teaches that separate operation.

**Exit check:** Choose types for an invoice quantity, unit price, postal code, and shipping qualification. State one business rule that the chosen type alone cannot enforce.


```{rubric} Footnotes
```

[^1]: Microsoft's [floating-point numeric types](https://learn.microsoft.com/en-us/dotnet/csharp/language-reference/builtin-types/floating-point-numeric-types) reference distinguishes binary floating-point and decimal precision.
[^2]: Microsoft defines `char` as a UTF-16 code unit in [The char type](https://learn.microsoft.com/en-us/dotnet/csharp/language-reference/builtin-types/char).
